In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
from scipy.stats import spearmanr

Feature extraction functions

In [5]:
def normalize_rna(sequence: str) -> str:
    value = str(sequence).upper().replace('T', 'U').replace(' ', '')
    if not value or any(base not in 'ACGU' for base in value):
        raise ValueError(f'Invalid RNA/DNA sequence: {sequence!r}')
    return value

def reverse_complement_rna(sequence: str) -> str:
    sequence = normalize_rna(sequence)
    complement = {'A': 'U', 'U': 'A', 'C': 'G', 'G': 'C'}
    return ''.join(complement[base] for base in reversed(sequence))

def find_mismatch_positions(spacer: str, target: str) -> list:
    spacer_rna = normalize_rna(spacer)
    target_rna = normalize_rna(target)
    if len(spacer_rna) != len(target_rna):
        raise ValueError(f'Length mismatch: {len(spacer_rna)} != {len(target_rna)}')
    return [i for i, (s, t) in enumerate(zip(spacer_rna, target_rna)) if s != t]

def _duplex_energy(spacer_rna: str, target_rna_same_orientation: str) -> float:
    import RNA
    pairing_strand = reverse_complement_rna(target_rna_same_orientation)
    return float(RNA.duplexfold(spacer_rna, pairing_strand).energy)

def extract_final_features_v4(spacer: str, target: str, mismatch_pos=None) -> list:
    spacer_rna = normalize_rna(spacer)
    target_rna = normalize_rna(target)
    if len(spacer_rna) != 20 or len(target_rna) != 20:
        raise ValueError('Cas12a spacer and protospacer must both be 20 nt')

    inferred = find_mismatch_positions(spacer_rna, target_rna)
    if mismatch_pos is not None and mismatch_pos >= 0:
        if mismatch_pos >= 20:
            raise ValueError(f'mismatch_pos out of range: {mismatch_pos}')
        if mismatch_pos not in inferred:
            raise ValueError(f'mismatch_pos={mismatch_pos} is not a mismatch')
        mismatch_positions = [mismatch_pos]
    else:
        mismatch_positions = inferred

    dg_duplex = _duplex_energy(spacer_rna, target_rna)
    dg_perfect = _duplex_energy(spacer_rna, spacer_rna)
    ddg = dg_duplex - dg_perfect if mismatch_positions else 0.0

    if not mismatch_positions:
        pos = -1.0
        is_purine_mismatch = 0.0
        is_transition = 0.0
        is_edge = 0.0
        mismatch_score = 0.0
        seed_weight = 1.0
    else:
        pos = float(min(mismatch_positions))
        spacer_base = spacer_rna[int(pos)]
        target_base = target_rna[int(pos)]
        is_purine_mismatch = float(spacer_base in 'AG' and target_base in 'AG')
        is_transition = float(
            (spacer_base, target_base) in {('A','G'),('G','A'),('C','U'),('U','C')}
        )
        is_edge = float(pos <= 2 or pos >= 17)
        mismatch_score = {
            ('A','C'): 3.0, ('C','A'): 3.0,
            ('G','A'): 2.0, ('A','G'): 2.0,
            ('C','U'): 1.0, ('U','C'): 1.0,
            ('G','U'): 0.0, ('U','G'): 0.0,
        }.get((spacer_base, target_base), 2.0)
        seed_weight = 1.5 if pos <= 8 else 1.0

    pos_ddg_interaction = pos * ddg if pos >= 0 else 0.0
    return [pos, ddg, is_purine_mismatch, is_transition,
            is_edge, pos_ddg_interaction, mismatch_score, seed_weight]

Load training data (all 50 samples from notebook)

In [6]:
print("Loading training data...")
data = [
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "AGGTTCGCGTGGATTAAAGG", "mismatch_pos": 0, "relative_activity": 1.244067},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "GGGTTCGCGTGGATTAAAGG", "mismatch_pos": 0, "relative_activity": 1.536652},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "TGGTTCGCGTGGATTAAAGG", "mismatch_pos": 0, "relative_activity": 1.349063},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CAGTTCGCGTGGATTAAAGG", "mismatch_pos": 1, "relative_activity": 1.002632},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CCGTTCGCGTGGATTAAAGG", "mismatch_pos": 1, "relative_activity": 1.327277},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CTGTTCGCGTGGATTAAAGG", "mismatch_pos": 1, "relative_activity": 1.293087},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGATTCGCGTGGATTAAAGG", "mismatch_pos": 2, "relative_activity": 1.108303},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGCTTCGCGTGGATTAAAGG", "mismatch_pos": 2, "relative_activity": 1.294277},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGTTTCGCGTGGATTAAAGG", "mismatch_pos": 2, "relative_activity": 1.230956},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGATCGCGTGGATTAAAGG", "mismatch_pos": 3, "relative_activity": 1.275075},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGCTCGCGTGGATTAAAGG", "mismatch_pos": 3, "relative_activity": 1.259007},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGGTCGCGTGGATTAAAGG", "mismatch_pos": 3, "relative_activity": 1.422551},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTACGCGTGGATTAAAGG", "mismatch_pos": 4, "relative_activity": 1.206470},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTCCGCGTGGATTAAAGG", "mismatch_pos": 4, "relative_activity": 1.287469},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTGCGCGTGGATTAAAGG", "mismatch_pos": 4, "relative_activity": 1.460275},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTAGCGTGGATTAAAGG", "mismatch_pos": 5, "relative_activity": 1.150048},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTGGCGTGGATTAAAGG", "mismatch_pos": 5, "relative_activity": 1.344187},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTTGCGTGGATTAAAGG", "mismatch_pos": 5, "relative_activity": 1.151763},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCACGTGGATTAAAGG", "mismatch_pos": 6, "relative_activity": 0.966740},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCCCGTGGATTAAAGG", "mismatch_pos": 6, "relative_activity": 1.109856},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCTCGTGGATTAAAGG", "mismatch_pos": 6, "relative_activity": 1.107226},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGAGTGGATTAAAGG", "mismatch_pos": 7, "relative_activity": 1.085175},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGGGTGGATTAAAGG", "mismatch_pos": 7, "relative_activity": 1.202152},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGTGTGGATTAAAGG", "mismatch_pos": 7, "relative_activity": 1.058207},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCATGGATTAAAGG", "mismatch_pos": 8, "relative_activity": 1.037646},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCCTGGATTAAAGG", "mismatch_pos": 8, "relative_activity": 1.133941},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCTTGGATTAAAGG", "mismatch_pos": 8, "relative_activity": 1.131164},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGAGGATTAAAGG", "mismatch_pos": 9, "relative_activity": 1.108452},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGCGGATTAAAGG", "mismatch_pos": 9, "relative_activity": 1.241383},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGGGGATTAAAGG", "mismatch_pos": 9, "relative_activity": 1.171693},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTAGATTAAAGG", "mismatch_pos": 10, "relative_activity": 1.054520},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTCGATTAAAGG", "mismatch_pos": 10, "relative_activity": 1.267676},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTTGATTAAAGG", "mismatch_pos": 10, "relative_activity": 1.222937},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTGAATTAAAGG", "mismatch_pos": 11, "relative_activity": 0.994914},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTGCATTAAAGG", "mismatch_pos": 11, "relative_activity": 1.275432},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTGTATTAAAGG", "mismatch_pos": 11, "relative_activity": 1.181162},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTGGCTTAAAGG", "mismatch_pos": 12, "relative_activity": 1.315327},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTGGGTTAAAGG", "mismatch_pos": 12, "relative_activity": 1.241909},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTGGTTTAAAGG", "mismatch_pos": 12, "relative_activity": 1.321736},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTGGAATAAAGG", "mismatch_pos": 13, "relative_activity": 1.173173},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTGGACTAAAGG", "mismatch_pos": 13, "relative_activity": 1.114124},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTGGAGTAAAGG", "mismatch_pos": 13, "relative_activity": 1.276508},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTGGATAAAAGG", "mismatch_pos": 14, "relative_activity": 1.171015},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTGGATCAAAGG", "mismatch_pos": 14, "relative_activity": 1.209477},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTGGATGAAAGG", "mismatch_pos": 14, "relative_activity": 1.376292},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTGGATTCAAGG", "mismatch_pos": 15, "relative_activity": 1.349096},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTGGATTGAAGG", "mismatch_pos": 15, "relative_activity": 1.274115},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTGGATTTAAGG", "mismatch_pos": 15, "relative_activity": 1.199478},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTGGATTACAGG", "mismatch_pos": 16, "relative_activity": 1.355353},
    {"spacer_seq": "CGGUUCGCGUGGAUUAAAGG", "target_seq": "CGGTTCGCGTGGATTAGAGG", "mismatch_pos": 16, "relative_activity": 1.332736},
]

df = pd.DataFrame(data)

print("Extracting features from training data...")
X_train = np.array([
    extract_final_features_v4(row['spacer_seq'], row['target_seq'], row['mismatch_pos'])
    for _, row in df.iterrows()
])
y_train = df['relative_activity'].values

print("Training models...")
svr = make_pipeline(StandardScaler(), SVR(kernel='rbf', C=20, epsilon=0.05, gamma=0.05))
gb = GradientBoostingRegressor(n_estimators=100, max_depth=2, learning_rate=0.05,
                                subsample=0.8, random_state=42)
svr.fit(X_train, y_train)
gb.fit(X_train, y_train)

Loading training data...
Extracting features from training data...
Training models...


UnicodeDecodeError: 'gbk' codec can't decode byte 0x94 in position 2952: illegal multibyte sequence

UnicodeDecodeError: 'gbk' codec can't decode byte 0x94 in position 2952: illegal multibyte sequence

GradientBoostingRegressor(learning_rate=0.05, max_depth=2, random_state=42,
                          subsample=0.8)

Load candidates

In [7]:
print("\nLoading candidates...")
candidates = pd.read_csv('E:/Evelyn/mirna-crispr-ml/candidate_sequences_corrected.csv')


Loading candidates...


Rename columns to match expected format

In [8]:
candidates.rename(columns={
    'crRNA_spacer_rna_5to3': 'crRNA_sequence_5to3',
    'target_mirna': 'target'
}, inplace=True)

Extract just the target label (29a, 29b, 29c)

In [9]:
candidates['target'] = candidates['target'].str.extract(r'miR-(\d+[abc])-3p')[0]
print(f"Loaded {len(candidates)} candidates")

Loaded 10 candidates


Define targets

In [10]:
MIR29A = 'UAGCACCAUCUGAAAUCGGUUA'
MIR29B = 'UAGCACCAUUUGAAAUCAGUGUU'
MIR29C = 'UAGCACCAUUUGAAAUCGGUUA'

targets = {
    '29a': MIR29A,
    '29b': MIR29B,
    '29c': MIR29C,
}

def find_mismatches(spacer: str, target: str) -> list:
    s = spacer.upper().replace('T', 'U')
    t = target.upper().replace('T', 'U')
    min_len = min(len(s), len(t))
    return [i for i in range(min_len) if s[i] != t[i]]

# Predict activity for each target
print("\nPredicting activities...")
for mirna_name, target_seq in targets.items():
    print(f"  Processing mir{mirna_name}...")
    features = []
    for _, row in candidates.iterrows():
        spacer = row['crRNA_sequence_5to3']
        spacer_20 = spacer[:20]
        target_20 = target_seq[:20]

        mm_list = find_mismatches(spacer_20, target_20)

        if len(mm_list) == 0:
            feat = extract_final_features_v4(spacer_20, target_20, -1)
        elif len(mm_list) == 1:
            feat = extract_final_features_v4(spacer_20, target_20, mm_list[0])
        else:
            mismatch_features = []
            for mm_pos in mm_list:
                feat = extract_final_features_v4(spacer_20, target_20, mm_pos)
                mismatch_features.append(feat)
            feat = np.mean(mismatch_features, axis=0)
        features.append(feat)

    X = np.array(features)
    candidates[f'activity_mir{mirna_name}'] = (svr.predict(X) + gb.predict(X)) / 2


Predicting activities...
  Processing mir29a...
  Processing mir29b...
  Processing mir29c...


Calculate selectivity ratio

In [11]:
print("\nCalculating selectivity ratios...")
def calculate_selectivity_ratio(row):
    target = row['target']
    activity_on_target = row[f'activity_mir{target}']

    other_targets = [t for t in ['29a', '29b', '29c'] if t != target]
    activity_off_target_max = max(
        row[f'activity_mir{other_targets[0]}'],
        row[f'activity_mir{other_targets[1]}']
    )

    eps = 1e-6
    return activity_on_target / (activity_off_target_max + eps)

candidates['selectivity_ratio'] = candidates.apply(calculate_selectivity_ratio, axis=1)

result = candidates.sort_values('selectivity_ratio', ascending=False)
result.to_csv('E:/Evelyn/ranked_crRNA_selectivity_v4_corrected.csv', index=False)

print(f'\n=== RESULTS ===')
print(f'Total candidates: {len(result)}')
print(f'Top candidate: {result.iloc[0]["candidate_id"]}')
print(f'\nTop 10 candidates by selectivity:')
print(result[['candidate_id', 'target', 'activity_mir29a', 'activity_mir29b',
              'activity_mir29c', 'selectivity_ratio']].head(10).to_string(index=False))

if result.iloc[0]['candidate_id'] == 'CRRNA-CORR-008':
    print('\nCRRNA-CORR-008 ranks first!')
else:
    print(f'\nTop candidate is {result.iloc[0]["candidate_id"]}, not CRRNA-CORR-008')


Calculating selectivity ratios...

=== RESULTS ===
Total candidates: 10
Top candidate: CRRNA-CORR-008

Top 10 candidates by selectivity:
  candidate_id target  activity_mir29a  activity_mir29b  activity_mir29c  selectivity_ratio
CRRNA-CORR-008    29c         1.191407         1.170097         1.536494           1.289646
CRRNA-CORR-004    29b         1.258956         1.536494         1.357519           1.131839
CRRNA-CORR-001    29a         1.536494         1.367524         1.228186           1.123558
CRRNA-CORR-010    29c         1.188043         1.131969         1.194016           1.005027
CRRNA-CORR-009    29c         1.194385         1.118711         1.192968           0.998813
CRRNA-CORR-005    29b         1.191902         1.162825         1.194724           0.973299
CRRNA-CORR-002    29a         1.191004         1.319781         1.218945           0.902424
CRRNA-CORR-007    29b         1.295001         1.163888         1.291396           0.898754
CRRNA-CORR-006    29b         1.27